# 🌳 Decision Tree — Placement Prediction

**Dataset:** `placement.csv` — 550 students
**Goal:** predict whether a student gets placed — `placed` = 1 (placed) or 0 (not placed)
**What we will do:** build a small tree → read it → test it → let it grow too much (overfitting) → find the right depth

## 📘 What is a Decision Tree?

### Definition

A **Decision Tree** is a supervised machine learning algorithm used for **classification** (and also regression). It predicts the answer by asking a series of **yes / no questions** about the features, like a flowchart.

- Each question splits the students into two smaller groups.
- The tree keeps splitting until a group is **pure** (all one class) or a stopping rule is reached (for example `max_depth`).
- To predict for a new student, we start at the top and follow the answers down to a **leaf**. The leaf gives the class.

### Parts of a tree

| Part | Meaning | In our placement tree |
|---|---|---|
| **Root node** | The first question, at the top | `cgpa <= 7.17` |
| **Internal node** | A question in the middle | `backlogs <= 1.5` |
| **Branch** | The answer to a question — left = True, right = False | yes / no |
| **Leaf node** | The end of a path — it gives the prediction | Placed / Not placed |
| **Depth** | The number of questions from the root to the deepest leaf | `max_depth = 2` |

### How the tree picks a question (what we did by hand)

1. **Entropy** measures how mixed a group is: **E = − p₁·log₂(p₁) − p₀·log₂(p₀)** → 0 for a pure group, 1 for a 50-50 group.
2. **Information Gain** = entropy of the parent − weighted entropy of the children.
3. The feature with the **highest information gain** becomes the question.
4. For a **numeric** column like `cgpa`: sort the values → take the midpoints → compute the gain for each midpoint → keep the best one as the **threshold**.

### Gini — what scikit-learn uses by default

scikit-learn measures impurity with **Gini** instead of entropy: **Gini = 1 − (p₁² + p₀²)**. It works the same way — 0 for a pure group, higher for a mixed group — and the tree still picks the split that reduces impurity the most. If you want the exact formula we used by hand, write `DecisionTreeClassifier(criterion='entropy')`.

### Key points

- **No scaling needed** — every question looks at one feature at a time.
- Easy to read and explain — the tree is a set of if-else rules.
- A tree that is allowed to grow too deep **memorises** the training data → **overfitting**. We control this with `max_depth`.
- In scikit-learn: `from sklearn.tree import DecisionTreeClassifier`

## 📥 1. Load the data

> **Flow:** import libraries → read the CSV → first look

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/Users/shailesh/Desktop/vedam_Sem3_ml/Data/placement .csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

## 🧹 2. Clean the data

> **Flow:** drop columns we will not use → fill missing values with the median

| Column | Why we drop it |
|---|---|
| `student_id` | Just a roll number — it says nothing about placement |
| `package_lpa` | Known only **after** placement (0 for every student who was not placed) — keeping it gives the answer away |
| `branch`, `hostel` | Text columns — dropped to keep the model numeric only |

In [ ]:
df = df.drop(columns=['student_id', 'package_lpa', 'branch', 'hostel'])
df.head()

In [ ]:
df['cgpa'] = df['cgpa'].fillna(df['cgpa'].median())
df['prep_hours'] = df['prep_hours'].fillna(df['prep_hours'].median())
df['communication_score'] = df['communication_score'].fillna(df['communication_score'].median())

df.isnull().sum()

## 🎯 3. Features (X) and target (y)

> **Flow:** X = every column except `placed` → y = `placed`

In [ ]:
X = df.drop(columns='placed')
y = df['placed']

X.columns

In [ ]:
y.value_counts()

## ✂️ 4. Train / test split

> **Flow:** 80% of students to train the tree → 20% kept aside to test it

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

## 📏 5. Do we need scaling? — No

> **Flow:** skip `StandardScaler` for trees

- KNN measures **distance** and logistic regression multiplies features by **weights**, so the size of the numbers matters — we scaled for them.
- A decision tree only asks **one question about one feature at a time**, like `cgpa <= 7.17` or `prep_hours <= 366`.
- Changing the units of `prep_hours` would only change the threshold number, not the split. So we give the tree the **raw data**.

## 🤖 6. Build a small tree (max_depth = 2)

> **Flow:** import `DecisionTreeClassifier` → create it with `max_depth=2` → `fit` on the training data

We start with only **2 levels of questions**, so the whole tree fits on the screen and we can read it.

## 🌳 7. Look at the tree

> **Flow:** `plot_tree` to draw it → `export_text` to read it as rules

### How to read one box

| Line in the box | Meaning | Root box |
|---|---|---|
| `cgpa <= 7.175` | The question. **True → go left**, False → go right | the best first question |
| `gini` | Impurity — 0 means pure | 0.491 |
| `samples` | Number of training students in this box | 440 |
| `value` | [not placed, placed] counts | [191, 249] |
| `class` | The majority class — the prediction if we stopped here | Placed |

- **Orange** boxes lean towards *Not placed*, **blue** boxes lean towards *Placed*. A darker colour means a purer box.
- The threshold **7.175** is the midpoint between two neighbouring CGPA values in the training data (7.17 and 7.18), the same midpoint method we used by hand. `export_text` below rounds it to 7.17.
- The tree picked `cgpa` as the root by itself — it reduces impurity the most, exactly like the highest information gain in our hand calculation.

## 🔮 8. Predict and check accuracy

> **Flow:** `predict` on the test data → `accuracy_score` on train and on test

## 🧮 9. Confusion matrix

> **Flow:** `confusion_matrix(y_test, y_pred)` → plot it with a seaborn heatmap

|  | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN — correctly said not placed | FP — said placed, but was not |
| **Actual 1** | FN — said not placed, but was placed | TP — correctly said placed |

## 🌲 11. Let the tree grow fully — overfitting

> **Flow:** create a tree with **no** `max_depth` → fit → compare train and test accuracy

Without a limit, the tree keeps asking questions until every leaf is pure.

### What happened?

- **Train accuracy = 1.00** — the tree is perfect on the students it has already seen.
- **Test accuracy ≈ 0.74** — on new students it is no better than our tiny 2-level tree (0.745). It is actually a little worse.
- With 13 levels and 82 leaves, many leaves hold only 1 or 2 students. Those questions describe **individual students (and their noise)**, not a general pattern.

This is **overfitting**: the model memorised the training data instead of learning from it.

## ⚖️ 13. Underfitting vs good fit vs overfitting

| | Underfitting | Good fit | Overfitting |
|---|---|---|---|
| **Model is** | too simple | just right | too complex |
| **Train accuracy** | low | high | very high (≈ 1.00) |
| **Test accuracy** | low | high | much lower than train |
| **Gap (train − test)** | small | small | **large** |
| **Our tree** | max_depth = 1 → 0.69 / 0.69 | max_depth = 3 → 0.78 / 0.78 | no limit → 1.00 / 0.74 |
| **Fix** | allow more depth | keep it | reduce `max_depth` |

- **Underfitting** — the model has not learned the pattern, so it does badly even on the training data. (Also called **high bias**.)
- **Overfitting** — the model memorised the training data, noise included, so it does badly on new data. (Also called **high variance**.)
- **Good fit** — test accuracy is as high as we can get **and** close to train accuracy.